# Debugging and Testing

<div class="alert alert-info">

### Aims:
 - Understand the importance of debugging
 - Learn some debugging techniques: reading tracebacks, static analysis, minimal failing examples
 - Learn the basics of `pdb`
 - Write and run tests with `pytest`, and turn every bug you fix into a test
 - Automatic documentation

<div class="alert alert-info">

### Notes

- Some of the material presented in today's lecture has been adapted from the [Scientific Python Lectures' debugging chapter](https://lectures.scientific-python.org/advanced/debugging/index.html).

- Scripts for today's lecture are in the `files` folder within `Lecture04`; the notes name each one as
  `files/<name>`. Run the commands in these notes from inside that folder (`cd Lecture04/files`).

Everything this lecture uses (`pytest`, `pytest-cov`, `flake8`, `sphinx` and `scikit-image`) is already in the module's `mpm2026` environment.

Building a PDF manual with Sphinx, at the very end of the lecture, is optional and also needs LaTeX, which is a large install. If you want to try it, run this in your Ubuntu (WSL) terminal:
```bash
sudo apt install texlive-latex-base texlive-latex-recommended texlive-latex-extra
```

When developing software it's often the case that you'll spend as much (if not more!) time debugging, compared to writing, 'actual' code. This statement alone should make one thing abundantly clear: debugging is important and in order to increase your productivity you _need_ to get good at it.

Over the course of time, as you develop/encounter code with bugs, fix them and repeat this process countless times, these are skills you will be 'forced' to acquire naturally. But as with everything, when starting out it doesn't hurt to get a few tips from sources who have spent too much time searching for that elusive typo in the middle of a million lines of code. Thus, this lecture aims to provide you with a few tips and tricks to help you on your journey to becoming a debugging guru.

## Debugging ≠ Fixing

**Debugging is the process of *understanding* why the system behaves as it does.**  
Fixing is the change you make *after* you understand the cause.

Why this matters:
- If you skip straight to “fixing,” you often patch symptoms and create regressions.
- A good debugging loop is: **Reproduce → Reduce → Inspect → Form a hypothesis → Test → Confirm → Fix → Add a test**.

Takeaway for today: we’ll prioritise tools and habits that maximise *understanding* quickly (tracebacks, `breakpoint()`, minimal reproducers, and tests that lock in the fix).

A funny salt gun advert...

<img src="figures/ldbp1.png" width=500 height=1000 />

This lectures advert...

<img src="figures/ldbp2.png" width=500 height=1000 />

## Debugging in Python

This lecture focuses on debugging in Python. While the general skill set is transferable to any coding language, some of the tools are not. Similar tools will however, generally, be available in other languages. For example, today we will see the Python debugger, `pdb`, in action. Later in the course, when some of you encounter `C++`, a comparable tool is the GNU debugger, `gdb`.  

### Infamous 'bugs'

There have been some infamous 'bugs' throughout history, here are a few:

#### _The world's first computer bug?_

On September 9th, 1947 at 3:45 p.m., operators of the Harvard Mark II computer, which Grace Murray Hopper worked on, found a moth trapped in a relay and taped it into their log book with the note "First actual case of bug being found". (Engineers already called faults "bugs"; Hopper helped make the story famous.) The term "bug" in computer science is now, of course, not taken literally. It is used to describe a flaw or failure in a computer program that causes it to produce an unexpected result or crash.

#### _The Y2K bug_

The *year 2000* problem spawned fears of worldwide economic collapse, nuclear war, and led to an industry of consultants providing last-minute fixes.

Scary stuff, but clearly exaggerated. Nevertheless, the bug was real and billions of dollars were spent worldwide fixing the bug. Indeed, not all systems were fixed in time and in Spain some parking meters failed, the French meteorological institute published the weather for 19100 and in Australia some ticket validation machines failed.

#### _The Dhahran Missile_

But on a more serious note, a bug in the software of an MIM-104 Patriot caused its system clock to drift by one third of a second over a period of one hundred hours - resulting in failure to locate and intercept an incoming Iraqi Al Hussein missile, which then struck Dhahran barracks, Saudi Arabia (February 25, 1991), killing 28 Americans.

And here are some of the more 'interesting' bugs you may encounter while developing software:

#### _The Heisenbug_

From [Wikipedia](https://en.wikipedia.org/wiki/Heisenbug#See_also):

_In computer programming jargon, a heisenbug is a software bug that seems to disappear or alter its behavior when one attempts to study it. The term is a pun on the name of Werner Heisenberg, the physicist who first asserted the observer effect of quantum mechanics, which states that the act of observing a system inevitably alters its state. In electronics the traditional term is probe effect, where attaching a test probe to a device changes its behavior._

#### _Mandelbug_

Mandelbugs do not change their properties or vanish like Heisenbugs. Instead, they are so unusual and complex that there is typically no practical solution to fix them. In fact, Mandelbugs may be so nondeterministic that many scholarly developers believe that fractal mathematics inventor Benoit Mandelbrot developed the Mandelbug to drive developers insane.

#### _Bohrbug_

A software bug which manifests reliably under a well-defined, but possibly unknown, set of conditions.

#### _Schroedinbug_

A software bug which manifests only when somebody debugging it finds out that it shouldn't work at all.

## Python tracebacks

Lets first focus on bugs that result in exceptions being raised in your python code - that is, your code doesn't complete and exits with a traceback. Initially, you'll probably spend a good deal of time fixing bugs of this form!

Lets consider a code snippet:

In [1]:
import numpy as np

def a(b):
    return np.sin(exp(b))

print(a(1))

NameError: name 'exp' is not defined

The first part of the error message reads:

```Python
---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
```

The first piece of the message reads `NameError`. This tells us the type of exception that has occurred. From Python's documentation:

```python
exception NameError

    Raised when a local or global name is not found. This applies only to unqualified names. The associated value is an error message that includes the name that could not be found.

    The name attribute can be set using a keyword-only argument to the constructor. When set it represent the name of the variable that was attempted to be accessed.
```

There we go. We're trying to use something in our code that we haven't defined!
Python has a large range of built-in exceptions that will generate tracebacks. See the documentation [here](https://docs.python.org/3/library/exceptions.html). Note that we can also define our own exceptions if required!

Let's continue analysing the traceback. Next it tells us `Traceback (most recent call last)` - that is, the last line shown in the traceback is where the exception occurred. This line may or may not be the source of the bug, but is often a good place to start looking. Hence, tracebacks should generally be read from 'bottom-to-top' when debugging. (For explanatory purposes, we'll read this one from top-to-bottom though).

Next, we have the start of the call sequence that resulted in the exception:

```python
Cell In[1], line 6
      2 
      3 def a(b):
      4     return np.sin(exp(b))
      5 
----> 6 print(a(1))
```

This is telling us that `print(a(1))` has been called successfully (since the traceback doesn't end there), along with the line it was called from and where that line is (`Cell In[1]` is the notebook cell; for a script, the traceback names the file instead). Since the exception occurs in a small snippet of code within a Jupyter notebook, the information is not 'super useful', but we'll see some cases later where it is.

Next we have

```python
Cell In[1], line 4, in a(b)
      3 def a(b):
----> 4     return np.sin(exp(b))

NameError: name 'exp' is not defined
```

Next, the traceback is telling us that on line 4, `exp` is not defined and is hence resulting in a `NameError` exception. The exponential function we're looking for is defined within `NumPy` and hence if we modify line 4 to `return np.sin(np.exp(b))` all should hopefully be good!

Now we've seen an example, let's put a slightly more rigorous definition of tracebacks in place. In Python, tracebacks are reports containing the function calls made in your code at a specific point. When an exception is raised, the traceback will provide information about what has gone wrong. Tracebacks are important, and very useful pieces of information to help you debug. In python, they tend to be more helpful than in many other coding languages!

#### Exercise

To understand tracebacks, we first need to understand what leads to the different exceptions we may encounter. Can you write some code snippets to generate the following exceptions:

1. `ImportError`
2. `ModuleNotFoundError`
3. `ZeroDivisionError`
4. `IndexError`
5. `TypeError`
6. `SyntaxError`
7. `IndentationError`
8. `RecursionError`

In [ ]:
# 1. ImportError


In [ ]:
# 2. ModuleNotFoundError


In [ ]:
# 3. ZeroDivisionError


In [ ]:
# 4. IndexError


In [ ]:
# 5. TypeError


In [ ]:
# 6. SyntaxError


In [ ]:
# 7. IndentationError


In [ ]:
# 8. RecursionError


## Fast static analysis

Static analysis tools read your code without running it. They catch whole classes of bugs (undefined names, unused imports, typos in names) before they turn up as a traceback, or worse, as a wrong answer. Some well-known ones for Python:
- [pyflakes](https://pypi.org/project/pyflakes/): fast and simple; detects syntax errors, undefined names and unused imports.
- [flake8](https://pypi.org/project/flake8/) (my personal favourite): `pyflakes`, plus `pycodestyle`, which checks your code against the [PEP 8](https://peps.python.org/pep-0008/) style guide, plus a complexity checker. It is in the `mpm2026` environment.
- [pylint](https://pypi.org/project/pylint/): slower, much more thorough, and more opinionated.
- [ruff](https://pypi.org/project/ruff/): a newer and very fast tool that implements the checks of `flake8` and many more. It is increasingly the default in new projects.

To check a notebook rather than a `.py` file, [nbQA](https://pypi.org/project/nbqa/) runs any of these on a notebook's code cells (`ruff` can check notebooks directly).

Integrating one of these into your editor or IDE (VS Code and JupyterLab both support this) is highly recommended: it does yield productivity gains.

#### Exercise: static analysis

From the `files` folder, run

```bash
flake8 .
```

Which of the messages point to code that is wrong or unnecessary (codes starting with `F`, from `pyflakes`), and which are about style (`E` and `W`, from `pycodestyle`)? Look up any code you don't recognise in the [`pyflakes` codes](https://flake8.pycqa.org/en/latest/user/error-codes.html) or the [`pycodestyle` codes](https://pycodestyle.pycqa.org/en/latest/intro.html#error-codes).

## Debugging workflow

If you do have a non trivial bug, this is when debugging strategies kick in. There is no silver bullet. Yet, strategies help:

For debugging a given problem, the favorable situation is when the problem is isolated in a small number of lines of code, outside framework or application code, with short modify-run-fail cycles
- Make it fail reliably. Find a test case that makes the code fail (in the same way) every time.
- Divide and Conquer. Once you have a failing test case, isolate the failing code.
    - Which module.
    - Which function.
    - Which line of code.

=> isolate a small reproducible failure: a test case/minimum failing example
- Change one thing at a time and re-run the failing test case.
- Use the debugger to understand what is going wrong.
- Take notes and be patient. It may take a while.

**Note**: Once you have gone through this process: isolated a tight piece of code reproducing the bug and fix the bug using this piece of code, add the corresponding code to your test suite.

## Minimum failing examples

Given the occurrence of a particular bug, a minimum failing example (MFE) is the simplest piece of code that can reproduce an instance of the particular bug in question (which was probably discovered through building a larger more complex script).

When approaching someone for help debugging a problem, it is considered good practice to provide them with an MFE. This will often drastically reduce the time taken for them to debug the problem. Further, when contributing to an open source project and reporting a problem (e.g. opening a bug report issue on Github), the developers will frequently ask 'Please could you provide an MFE'.

#### Example:

This is a 'real' example. (Don't worry about the details, this is just to illustrate and idea). I was once sent this long bit of code:

```python
import numpy                   as np
import matplotlib.pyplot       as plot
import math                    as mt
import matplotlib.ticker       as mticker    
from   mpl_toolkits.axes_grid1 import make_axes_locatable
from   matplotlib              import cm
from   scipy.interpolate       import CubicSpline
from   scipy.interpolate       import interp1d
from   devito import *

from   examples.seismic        import TimeAxis
from   examples.seismic        import RickerSource
from   examples.seismic        import Receiver

nptx   = 861
nptz   = 311
x0     = 13650.
x1     = 56650. 
compx  = x1-x0
z0     = 0.
z1     = 9920.
compz  = z1-z0;
hxv    = (x1-x0)/(nptx-1)
hzv    = (z1-z0)/(nptz-1)

npmlx  = 30
npmlz  = 31

lx = npmlx*hxv
lz = npmlz*hzv

origin  = (x0,z0)
extent  = (compx,compz)
shape   = (nptx,nptz)
spacing = (hxv,hzv)

class d0domain(SubDomain):
    name = 'd0'
    def define(self, dimensions):
        x, z = dimensions
        return {x: ('middle', npmlx, npmlx), z: ('middle', 0, npmlz)}
d0_domain = d0domain()

class d1domain(SubDomain):
    name = 'd1'
    def define(self, dimensions):
        x, z = dimensions
        return {x: ('left',npmlx), z: z}
d1_domain = d1domain()

class d2domain(SubDomain):
    name = 'd2'
    def define(self, dimensions):
        x, z = dimensions
        return {x: ('right',npmlx), z: z}
d2_domain = d2domain()

class d3domain(SubDomain):
    name = 'd3'
    def define(self, dimensions):
        x, z = dimensions
        return {x: ('middle', npmlx, npmlx), z: ('right',nptz)}
d3_domain = d3domain()

grid = Grid(origin=origin,extent=extent,shape=shape,subdomains=(d0_domain,d1_domain,d2_domain,d3_domain),dtype=np.float64) 

nptxvel = 1407
nptzvel = 311
x0vel   = 0.        
x1vel   = 70300.     
z0vel   = 0.        
z1vel   = 9920.
hxvel   = (x1vel-x0vel)/(nptxvel-1)
hzvel   = (z1vel-z0vel)/(nptzvel-1)
Xvel    = np.linspace(x0vel,x1vel,nptxvel)
Zvel    = np.linspace(z0vel,z1vel,nptzvel)
velarq = np.fromfile('velocity_data/profilevel.bin',dtype='float32')
velarq = velarq.reshape((nptxvel,nptzvel))
fscale = 10**(-3) 
velarq = fscale*velarq

X0 = np.linspace(x0,x1,nptx)   
Z0 = np.linspace(z0,z1,nptz)
X1 = np.linspace((x0+0.5*hxv),(x1-0.5*hxv),nptx-1)
Z1 = np.linspace((z0+0.5*hzv),(z1-0.5*hzv),nptz-1)
v0 = np.zeros((nptx,nptz))
v1 = np.zeros((nptx-1,nptz-1))

C0x = np.zeros((nptx,nptzvel))
for j in range(nptzvel):
    x = Xvel
    z = velarq[0:nptxvel,j]
    #cs = CubicSpline(x,z)    
    cs = interp1d(x,z,kind='linear')
    xs = X0
    C0x[0:nptx,j] = cs(xs)
    
for i in range(nptx):
    x = Zvel
    z = C0x[i,0:nptzvel]
    #cs = CubicSpline(x,z)
    cs = interp1d(x,z,kind='linear')
    xs = Z0
    v0[i,0:nptz] = cs(xs)
          
C11x = np.zeros((nptx-1,nptzvel))
for j in range(nptzvel):
    x = Xvel
    z = velarq[0:nptxvel,j]
    cs = CubicSpline(x,z)
    xs = X1
    C11x[0:nptx-1,j] = cs(xs)
    
for i in range(nptx-1):
    x  = Zvel
    z  = C11x[i,0:nptzvel]
    #cs = CubicSpline(x,z)
    cs = interp1d(x,z,kind='linear')
    xs = Z1
    v1[i,0:nptz-1] = cs(xs)

t0 = 0.
tn = 20000.   
CFL = 0.4
vmax  = np.amax(v0) 
dtmax = np.float64((min(hxv,hzv)*CFL)/(vmax))
ntmax = int((tn-t0)/dtmax)+1
dt0   = np.float64((tn-t0)/ntmax)

time_range = TimeAxis(start=t0,stop=tn,num=ntmax+1)
nt         = time_range.num - 1

(hx,hz) = grid.spacing_map  
(x, z)  = grid.dimensions     
t       = grid.stepping_dim
dt      = grid.stepping_dim.spacing

f0     = 0.005
nfonte = 1
xposf  = 35150.
zposf  = 32.

src = RickerSource(name='src',grid=grid,f0=f0,npoint=nfonte,time_range=time_range,staggered=NODE,dtype=np.float64)
src.coordinates.data[:, 0] = xposf
src.coordinates.data[:, 1] = zposf

nrec   = nptx
nxpos  = np.linspace(x0,x1,nrec)
nzpos  = 32.

rec = Receiver(name='rec',grid=grid,npoint=nrec,time_range=time_range,staggered=NODE,dtype=np.float64)
rec.coordinates.data[:, 0] = nxpos
rec.coordinates.data[:, 1] = nzpos

u = TimeFunction(name="u",grid=grid,time_order=2,space_order=2,staggered=NODE,dtype=np.float64)

phi1 = TimeFunction(name="phi1",grid=grid,time_order=2,space_order=2,staggered=(x,z),dtype=np.float64)
phi2 = TimeFunction(name="phi2",grid=grid,time_order=2,space_order=2,staggered=(x,z),dtype=np.float64)

vel0 = Function(name="vel0",grid=grid,space_order=2,staggered=NODE,dtype=np.float64)
vel0.data[:,:] = v0[:,:]

vel1 = Function(name="vel1", grid=grid,space_order=2,staggered=(x,z),dtype=np.float64)
vel1.data[0:nptx-1,0:nptz-1] = v1

vel1.data[nptx-1,0:nptz-1] = vel1.data[nptx-2,0:nptz-1]
vel1.data[0:nptx,nptz-1]   = vel1.data[0:nptx,nptz-2]

src_term = src.inject(field=u.forward,expr=src*dt**2*vel0**2)

rec_term = rec.interpolate(expr=u)

x0pml  = x0 + npmlx*hxv 
x1pml  = x1 - npmlx*hxv 
z0pml  = z0            
z1pml  = z1 - npmlz*hzv 

def fdamp(x,z,i):
    
    quibar  = 0.05
          
    if(i==1):
        a = np.where(x<=x0pml,(np.abs(x-x0pml)/lx),np.where(x>=x1pml,(np.abs(x-x1pml)/lx),0.))
        fdamp = quibar*(a-(1./(2.*np.pi))*np.sin(2.*np.pi*a))
    if(i==2):
        a = np.where(z<=z0pml,(np.abs(z-z0pml)/lz),np.where(z>=z1pml,(np.abs(z-z1pml)/lz),0.))
        fdamp = quibar*(a-(1./(2.*np.pi))*np.sin(2.*np.pi*a))
      
    return fdamp

def generatemdamp():
    
    X0     = np.linspace(x0,x1,nptx)    
    Z0     = np.linspace(z0,z1,nptz)
    X0grid,Z0grid = np.meshgrid(X0,Z0)
    X1   = np.linspace((x0+0.5*hxv),(x1-0.5*hxv),nptx-1)
    Z1   = np.linspace((z0+0.5*hzv),(z1-0.5*hzv),nptz-1)
    X1grid,Z1grid = np.meshgrid(X1,Z1)
   
    D01 = np.zeros((nptx,nptz))
    D02 = np.zeros((nptx,nptz))
    D11 = np.zeros((nptx,nptz))
    D12 = np.zeros((nptx,nptz))
    
    D01 = np.transpose(fdamp(X0grid,Z0grid,1))
    D02 = np.transpose(fdamp(X0grid,Z0grid,2))
  
    D11 = np.transpose(fdamp(X1grid,Z1grid,1))
    D12 = np.transpose(fdamp(X1grid,Z1grid,2))
    
    return D01, D02, D11, D12

D01, D02, D11, D12 = generatemdamp();

dampx0 = Function(name="dampx0", grid=grid,space_order=2,staggered=NODE ,dtype=np.float64)
dampz0 = Function(name="dampz0", grid=grid,space_order=2,staggered=NODE ,dtype=np.float64)
dampx0.data[:,:] = D01
dampz0.data[:,:] = D02

dampx1 = Function(name="dampx1", grid=grid,space_order=2,staggered=(x,z),dtype=np.float64)
dampz1 = Function(name="dampz1", grid=grid,space_order=2,staggered=(x,z),dtype=np.float64)
dampx1.data[0:nptx-1,0:nptz-1] = D11
dampz1.data[0:nptx-1,0:nptz-1] = D12

dampx1.data[nptx-1,0:nptz-1]   = dampx1.data[nptx-2,0:nptz-1]
dampx1.data[0:nptx,nptz-1]     = dampx1.data[0:nptx,nptz-2]
dampz1.data[nptx-1,0:nptz-1]   = dampz1.data[nptx-2,0:nptz-1]
dampz1.data[0:nptx,nptz-1]     = dampz1.data[0:nptx,nptz-2]

# White Region
pde01   = Eq(u.dt2-u.laplace*vel0*vel0) 

# Blue Region
pde02a  = u.dt2   + (dampx0+dampz0)*u.dtc + (dampx0*dampz0)*u - u.laplace*vel0*vel0 
pde02b  = - (0.5/hx)*(phi1[t,x,z-1]+phi1[t,x,z]-phi1[t,x-1,z-1]-phi1[t,x-1,z])
pde02c  = - (0.5/hz)*(phi2[t,x-1,z]+phi2[t,x,z]-phi2[t,x-1,z-1]-phi2[t,x,z-1])
pde02   = Eq(pde02a + pde02b + pde02c)

pde10 = phi1.dt + dampx1*0.5*(phi1.forward+phi1)
a1    = u[t+1,x+1,z] + u[t+1,x+1,z+1] - u[t+1,x,z] - u[t+1,x,z+1] 
a2    = u[t,x+1,z]   + u[t,x+1,z+1]   - u[t,x,z]   - u[t,x,z+1] 
pde11 = -(dampz1-dampx1)*0.5*(0.5/hx)*(a1+a2)*vel1*vel1
pde1  = Eq(pde10+pde11)
                                                    
pde20 = phi2.dt + dampz1*0.5*(phi2.forward+phi2) 
b1    = u[t+1,x,z+1] + u[t+1,x+1,z+1] - u[t+1,x,z] - u[t+1,x+1,z] 
b2    = u[t,x,z+1]   + u[t,x+1,z+1]   - u[t,x,z]   - u[t,x+1,z] 
pde21 = -(dampx1-dampz1)*0.5*(0.5/hz)*(b1+b2)*vel1*vel1
pde2  = Eq(pde20+pde21)

stencil01 =  Eq(u.forward,solve(pde01,u.forward) ,subdomain = grid.subdomains['d0'])

subds = ['d1','d2','d3']

stencil02 = [Eq(u.forward,solve(pde02, u.forward),subdomain = grid.subdomains[subds[i]]) for i in range(0,len(subds))]
stencil1 = [Eq(phi1.forward, solve(pde1,phi1.forward),subdomain = grid.subdomains[subds[i]]) for i in range(0,len(subds))]
stencil2 = [Eq(phi2.forward, solve(pde2,phi2.forward),subdomain = grid.subdomains[subds[i]]) for i in range(0,len(subds))]

bc  = [Eq(u[t+1,0,z],0.),Eq(u[t+1,nptx-1,z],0.),Eq(u[t+1,x,nptz-1],0.),Eq(u[t+1,x,0],u[t+1,x,1])]

op = Operator([stencil01,stencil02] + src_term + bc + [stencil1,stencil2] + rec_term,subs=grid.spacing_map)

u.data[:]     = 0.
phi1.data[:]  = 0.
phi2.data[:]  = 0.

op(time=nt,dt=dt0)

def graph2d(U):    
    plot.figure()
    fscale =  10**(-3)
    scale = np.amax(U[npmlx:-npmlx,0:-npmlz])/50.
    extent = [fscale*x0pml,fscale*x1pml,fscale*z1pml,fscale*z0pml]
    fig = plot.imshow(np.transpose(U[npmlx:-npmlx,0:-npmlz]),vmin=-scale, vmax=scale, cmap=cm.gray, extent=extent)
    plot.gca().xaxis.set_major_formatter(mticker.FormatStrFormatter('%d km'))
    plot.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%d km'))
    plot.axis('equal')
    plot.title('Map - Acoustic Problem with PML - Devito')
    plot.grid()
    ax = plot.gca()
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = plot.colorbar(fig, cax=cax, format='%.2e')
    cbar.set_label('Displacement [km]')
    plot.draw()
    plot.savefig('acoustic_map_pml_devito.png',dpi=100)
    plot.show()
    
graph2d(u.data[0,:,:])

def graph2drec(rec):    
        fig1 = plot.figure()
        fscale =  10**(-3)
        scale = np.amax(rec[:,npmlx:-npmlx])/50.
        extent = [fscale*x0pml,fscale*x1pml, fscale*tn, fscale*t0]
        fig = plot.imshow(rec[:,npmlx:-npmlx], vmin=-scale, vmax=scale, cmap=cm.gray, extent=extent)
        plot.gca().xaxis.set_major_formatter(mticker.FormatStrFormatter('%d km'))
        plot.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%d s'))
        plot.axis('equal')
        plot.title('Receivers Signal Profile with PML - Devito')
        ax = plot.gca()
        divider = make_axes_locatable(ax)
        plot.savefig('rec_map_pml_devito.png')
        plot.show()

graph2drec(rec.data)
```

which was now producing an incorrect result, but a couple of software versions ago the result was correct.

The above piece of code is long and the offender has not been pin-pointed, thus making it difficult to deduce where the bug is coming from. After some work, I managed to produce an MFE:

```python
import numpy as np
from devito import *
class d0domain(SubDomain):
    name = 'd0'
    def define(self, dimensions):
        x, y = dimensions
        return {x: ('middle', 2, 2), y: ('right', 10)}
sdom = d0domain()
grid = Grid(shape=(10, 10), extent=(10, 10), subdomains=(sdom))
u = Function(name='u', grid=grid)
eq = Eq(u, u+1, subdomain = grid.subdomains['d0'])
op = Operator(eq)
op.apply()
```

This piece of code is short and simple and allowed the bug to be tracked down and fixed in less than an hour!

## Manual debugging

Here are a couple of 'manual' debugging techniques:

- Print statements
Adding print statements to your code can often be a quick and effective method of pin-pointing problematic lines of code, e.g.,
```python
...
x = some_function(inputs)
print(x.property1, x.property2)
...
```

- `from IPython import embed; embed()`
When possible, this command can be an incredibly useful tool for debugging sections of code. It is designed for terminal sessions (a script, or `IPython`); in a notebook, `breakpoint()`, which we meet below, works better:
```python
...
x = some_function(inputs)
from IPython import embed; embed()
...
```

I'll now show you a live example, using `files/fourier_series.py`.

#### Code for a simple Fourier series

Let $f$ be the function with period $2\pi$ defined on the interval $[-\pi,\pi]$ by:

\begin{equation*}
  f(x)=x^2
\end{equation*}

This function is shown below:

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

N = 1000
x = np.linspace(-np.pi,np.pi,N)

fig=plt.figure(figsize=(12,8), dpi= 60, facecolor='w', edgecolor='k')
plt.plot(x, x**2, label = "$y(x)=x^2$")
plt.xlabel("x")
plt.ylabel("y")
plt.ylim((-1,10))
plt.legend()
plt.show()

The Fourier series of $x^2$ is:

\begin{align*}
  f(x) &= \frac{a_0}{2}+\sum_{n=1}^{\infty}a_n\cos nx+\sum_{n=1}^{\infty}b_n\sin nx \\
       &= \frac{\pi^2}{3}+4\sum_{n=1}^{\infty}(-1)^n\frac{\cos nx}{n^2} \qquad
  |x|<\pi
\end{align*}

The script `files/fourier_series.py` plots the partial sums of this series for $N = 1, 2, 5, 10$ and $100$ terms, together with $x^2$ itself. Open it in VS Code and run it, or run it from the `files` folder:

```bash
python fourier_series.py
```

Oh dear, the result is all wrong...

Let's look at how we could use print statements, `breakpoint()` and `from IPython import embed; embed()` in `files/fourier_series.py` to find the problem. The comment in `eval_fc` marks a good place to start. (In a notebook, print statements and `breakpoint()` work in a code cell in the same way.)

## Using the Python debugger

The python debugger, pdb: https://docs.python.org/library/pdb.html, allows you to inspect your code interactively.

Specifically it allows you to:
- View the source code.
- Walk up and down the call stack.
- Inspect values of variables.
- Modify values of variables.
- Set breakpoints.

### Quick reference: common `pdb` commands

Below are the most useful commands you'll use 90% of the time when debugging interactively:

| Command | Meaning | Typical use |
|----------|----------|-------------|
| `h` | help | Show all commands or `h <command>` for details |
| `n` | next | Step to the next line in the *current* function |
| `s` | step | Step *into* a function call |
| `c` | continue | Continue execution until next breakpoint or end |
| `l` | list | Show source code around the current line |
| `p <expr>` | print | Evaluate and print an expression (e.g. `p x`) |
| `pp <expr>` | pretty-print | Same as `p`, but formatted nicely |
| `b <line>` | breakpoint | Set a breakpoint (e.g. `b 42`) |
| `b <file>:<line>` | breakpoint (by file) | e.g. `b myscript.py:20` |
| `cl` | clear | Remove all breakpoints |
| `u` / `d` | up / down | Move up or down the call stack |
| `q` | quit | Quit the debugger immediately |

**Note:**  
Since Python 3.7, you can simply call `breakpoint()` instead of  
`import pdb; pdb.set_trace()`.  
This respects the `PYTHONBREAKPOINT` environment variable and integrates well with tools like `pytest --pdb`.


### Invoking the debugger

Ways to launch the debugger:
1. Postmortem, launch debugger after module errors.
2. Launch the module with the debugger.
3. Call the debugger inside the module

#### Postmortem

**Example 1**: You're working in `IPython` and you get a traceback.

Here we debug `files/index_error.py`. Running it in `IPython` from the `files` folder (`%run index_error.py`) raises an `IndexError`: type `%debug` to drop into the debugger.

```bash
---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
File .../Lecture04/files/index_error.py:8
      5     print(lst[len(lst)])
      7 if __name__ == '__main__':
----> 8     index_error()

File .../Lecture04/files/index_error.py:5, in index_error()
      3 def index_error():
      4     lst = list('foobar')
----> 5     print(lst[len(lst)])

IndexError: list index out of range
```
-------------------------------------------------------------------

**Example 2:** Post-mortem debugging without IPython

In some situations you cannot use IPython, for instance to debug a script that wants to be called from the command line. In this case, you can call the script with `python -m pdb script.py`. This will result in `pdb` being launched at the beginning of your script. You can then execute via `continue` (or `run`). The script will then continue until an exception is encountered:

```bash
$ python -m pdb index_error.py 
> .../Lecture04/files/index_error.py(1)<module>()
-> """Small snippet to raise an IndexError."""
(Pdb) continue
Traceback (most recent call last):
  File ".../envs/mpm2026/lib/python3.14/pdb.py", line 3656, in main
    pdb._run(target)
    ~~~~~~~~^^^^^^^^
  File ".../envs/mpm2026/lib/python3.14/pdb.py", line 2566, in _run
    self.run(target.code)
    ~~~~~~~~^^^^^^^^^^^^^
  File ".../envs/mpm2026/lib/python3.14/bdb.py", line 913, in run
    exec(cmd, globals, locals)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File ".../Lecture04/files/index_error.py", line 8, in <module>
    index_error()
    ~~~~~~~~~~~^^
  File ".../Lecture04/files/index_error.py", line 5, in index_error
    print(lst[len(lst)])
          ~~~^^^^^^^^^^
IndexError: list index out of range
Uncaught exception. Entering post mortem debugging
Running 'cont' or 'step' will restart the program
> .../Lecture04/files/index_error.py(5)index_error()
-> print(lst[len(lst)])
(Pdb)
```
-------------------------------------------------------------------

#### Step-by-step execution

**Example**: You believe a bug exists in a module but are not sure where.

For instance we are trying to debug `files/wiener_filtering.py`. Indeed the code runs, but the filtering does not work well.

```bash
%run wiener_filtering.py
.../Lecture04/files/wiener_filtering.py:40: RuntimeWarning: divide by zero encountered in divide
  noise_level = (1 - noise/l_var )
```

- Run the script in `IPython` with the debugger using `%run -d wiener_filtering.py`  (or `python -m pdb wiener_filtering.py`):
-------------------------------------------------------------------
```bash
In [1]: %run -d wiener_filtering.py
*** Blank or comment
*** Blank or comment
*** Blank or comment
NOTE: Enter 'c' at the ipdb>  prompt to continue execution.
> .../Lecture04/files/wiener_filtering.py(1)<module>()
----> 1 """ Wiener filtering a noisy image (scikit-image's astronaut): this module is buggy
      2 """
      3 import numpy as np
      4 import scipy as sp
      5 import matplotlib.pyplot as plt

ipdb>
```
-------------------------------------------------------------------
- Set a break point at line 34 using `b 34`:
-------------------------------------------------------------------
```bash
ipdb> b 34
Breakpoint 1 at .../Lecture04/files/wiener_filtering.py:34
```
-------------------------------------------------------------------
- Continue execution to next breakpoint with `c(ont(inue))`:
-------------------------------------------------------------------
```bash
ipdb> c
> .../Lecture04/files/wiener_filtering.py(34)iterated_wiener()
     32         Do not use this: this is crappy code to demo bugs!
     33     """
1--> 34     noisy_img = noisy_img
     35     denoised_img = local_mean(noisy_img, size=size)
     36     l_var = local_var(noisy_img, size=size)
```
-------------------------------------------------------------------
- Step into code with `n(ext)` and `s(tep)`: next jumps to the next statement in the current execution context, while step will go across execution contexts, i.e. enable exploring inside function calls. After a few `n`'s and `s`'s, we'll get here:
-------------------------------------------------------------------
```bash
.../Lecture04/files/wiener_filtering.py:40: RuntimeWarning: divide by zero encountered in divide
  noise_level = (1 - noise/l_var )
> .../Lecture04/files/wiener_filtering.py(41)iterated_wiener()
     39         noise = (res**2).sum()/res.size
     40         noise_level = (1 - noise/l_var )
---> 41         noise_level[noise_level<0] = 0
     42         denoised_img = denoised_img + noise_level*res
     43     return denoised_img
```
-------------------------------------------------------------------
We then see our runtime warning pop-up and can go about exploring our local variables. E.g. let's look at `l_var`.

Oh dear, nothing but integers. Here is our bug, we are doing integer arithmetic. And indeed, if we look at `np.abs(l_var.min())` we have zeros.

### Other ways of starting a debugger

#### _Raising an exception as a poor man's break point_

If you find it tedious to note the line number to set a break point, you can simply raise an exception at the point that you want to inspect and use `IPython`’s `%debug`. Note that in this case you cannot step or continue the execution.

#### _Debugging test failures in_ `pytest`

Running, e.g., `pytest --pdb test_pdb_example.py` drops you into the debugger at the point where a test fails, whether through a failed `assert` or an unexpected exception, so you can inspect the failing test's variables. `pytest --trace` instead stops at the start of each test, so you can step through it from the beginning. (Because `pytest` captures what your code prints, `embed()` doesn't work well inside a test; `breakpoint()` and `--pdb` do.)

`files/test_pdb_example.py` contains a deliberately failing test: we'll run it after introducing `pytest`.

#### _Calling the debugger explicitly_

Insert the following line where you want to drop in the debugger:

```python
breakpoint()
```

This also works within notebooks, where the debugger's prompt appears below the cell.

We'll look at an additional example: `files/breakpoint_demo.py` calls `breakpoint()` inside a small function, `apply_discount`. Run it from the `files` folder (`python breakpoint_demo.py`), print `price`, `discount` and `discounted_price` at the `(Pdb)` prompt (e.g. `p discounted_price`), then type `c` to continue.

### Graphical debuggers and alternatives

- JupyterLab has a built-in visual debugger (the bug icon in a notebook's toolbar): set breakpoints by clicking next to line numbers, and inspect variables in a side panel.
- The Visual Studio Code integrated development environment includes a debugging mode.
- `pudb` is a good semi-graphical debugger with a text user interface in the console.

<div class="alert alert-danger" role="alert">

### Debugger commands are not Python code

At the `(Pdb)` prompt, anything that looks like a debugger command is treated as one. So if a variable has the same name as a command (`n`, `s`, `c`, `l`, `p`, `b`, `q`, ...), typing its name runs the command instead of showing the variable: typing `c` *continues* the program rather than printing `c`. Use `p c` to print it, or start the line with `!` to run it as Python (e.g. `!c = 0`). This is different from dropping into an embedded `IPython` shell, where everything you type is Python.

### Getting help when in the debugger

Type `h` or `help` to access the interactive help:
```bash
ipdb> help

Documented commands (type help <topic>):
========================================
EOF        cont        help           pdef     retval           unignore_module
a          context     ignore         pdoc     run              unt            
alias      continue    ignore_module  pfile    rv               until          
args       d           j              pinfo    s                up             
b          debug       jump           pinfo2   skip_hidden      w              
break      disable     l              pp       skip_predicates  whatis         
bt         display     list           psource  source           where          
c          down        ll             q        step           
cl         enable      longlist       quit     tbreak         
clear      exceptions  n              r        u              
commands   exit        next           restart  unalias        
condition  h           p              return   undisplay      

Miscellaneous help topics:
==========================
exec  pdb

Undocumented commands:
======================
interact
```

<div class="alert alert-success" role="alert">

### Exercise: debugging with `pdb`

The script `files/to_debug.py` compares four of SciPy's root-finding routines (`brenth`, `brentq`, `ridder` and `bisect`) by counting how many function evaluations each needs to find the roots of five functions, from many random starting brackets. It is documented and hopefully legible, and it seeks to answer a question of real interest in numerical computing, but it does not work.

1. Run it (`python to_debug.py` in the `files` folder), read the traceback, and use `pdb` (`python -m pdb to_debug.py`) to find and fix the bug.
2. Once the script runs, check that its output makes sense: there is a second, quieter bug. Find and fix it too.
3. Following the debugging loop at the start of this lecture, write a test that fails because of the second bug and passes once it is fixed. (`pytest`'s built-in [`capsys`](https://docs.pytest.org/en/stable/how-to/capture-stdout-stderr.html) fixture lets a test read what a function prints.)

</div>

<div class="alert alert-success" role="alert">

### Exercise: MFE

The script `files/mfe.py` fits a straight line to five measurements and prints the root-mean-square (RMS) error of the fit. It runs without any error, but the RMS error it reports is far too large for points that lie so close to a line. Find the problem, distil it into a minimal failing example of no more than a few lines, fix it, and write a test that would catch the bug if it came back.

</div>

## Avoiding bugs

- We all, even the great *Fabio Luporini*, write buggy code.
- Keep It Simple, Stupid (KISS):
    - What is the simplest thing that could possibly work?
- Don't Repeat Yourself (DRY):
    - Every piece of knowledge must have a single, unambiguous, authoritative representation within a system.
    - Constants, algorithms, etc.
- Try to limit interdependencies of your code. (Loose Coupling).
- Give your variables, functions and modules meaningful names (not cryptic/mathematics names).
- **Ensure your code is well tested!**
- **Ensure code is documented and that the documentation is correct.**

We'll look at some additional examples of these principles in action: `files/kiss.py`, `files/dry.py` and `files/clear_naming.ipynb`.

## Testing

Before diving into the types of tests, let's briefly introduce the tool we'll use throughout: **pytest**.

`pytest` is the de facto standard framework for testing in Python. It’s lightweight, doesn’t require much setup, and integrates easily with modern editors.

Writing tests in `pytest` is just like writing normal Python functions (and, when needed, classes), which check results with plain `assert` statements.  
If a file or function name begins with `test_`, `pytest` will automatically discover and run it — no boilerplate required.

We’ll see examples of this shortly.

### The many sorts of test

When developing software, opportunities (and often the need) to test appear at many different levels, and these families have developed their own names.

#### Unit tests

Unit tests are small and (ideally) quick tests which verify the behaviour of a single programming "unit" (i.e. a module or function). Thus they ensure that the unit satisfies the "contract" it makes in isolation.

Since the function is tested by itself, rather than on the job, inputs must be "mocked up" (i.e. hard-coded in the test, or replaced with trivial substitutes) to generate known outputs and external dependencies may be replaced with "test stubs" which short circuit to quickly give the information needed by the unit being tested. Let's look at the example in the `files/unittest` folder: `files/unittest/foo.py` defines `solve_quadratic`, and `files/unittest/test_quadratic_unit.py` tests it. The test file has one test function per behaviour (two roots, one root, no real roots, ...): when a test fails, its name tells you which behaviour broke.

We can invoke this test by executing the `pytest` command and pointing at the appropriate folder. (Note that `pytest` always searches recursively unless you tell it otherwise with the [norecursedirs](https://docs.pytest.org/en/stable/reference/reference.html#confval-norecursedirs) option).

For example, if we're in the files directory
```python
pytest unittest/
```
would do the job.

#### Integration tests

Tests which combine multiple program units together and confirm that the interaction proceeds as expected. This might mean chaining multiple functions you write together, or using the real external dependencies rather than the fakes you generated for your unit tests. Integration tests tend to exercise more code at once, but are slower to run and can be difficult to write effectively.

#### Feature/Functionality tests

Tests which confirm an entire feature is working successfully as a whole, effectively interacting with your software as a user would. For numerical codes, this often involves analytic solutions and/or using simple "toy" problems which are well understood and have nice solutions. For more general software, it might even involve testing human interaction to make sure that GUIs are clear and robust.

In larger projects, such tests generally exercise many, many lines of code at once.

#### Regression tests

Tests which check that fixed bugs stay fixed. I.e. that introducing a new change does not break the existing functionality. The tests may be at the level of the unit, integration or features. If bug reports have a minimal example attached, then this can provide the basis for useful examples for regression testing (this is a form of "red-green" testing similar to test driven development). This is the last step of the debugging loop at the start of this lecture: every bug you fix should leave a test behind.

### Checking floating-point results and exceptions

Two kinds of check come up all the time. First, floating-point results are rarely exact (Lecture 7 explains why), so comparing them with `==` makes tests fail for no good reason. `pytest.approx` compares to within a relative tolerance ($10^{-6}$ by default), and for NumPy arrays `np.testing.assert_allclose` does the same element by element:

In [ ]:
import numpy as np
import pytest

print(0.1 + 0.2 == 0.3)                  # False: rounding error
print(0.1 + 0.2 == pytest.approx(0.3))   # True: equal to within a relative tolerance

np.testing.assert_allclose(np.array([0.1 + 0.2, 1.0]), np.array([0.3, 1.0]))   # passes silently

Second, a function should often *refuse* bad input by raising an exception, and that behaviour deserves a test too. `pytest.raises` checks that the code inside the `with` block raises the given exception, and the test fails if it doesn't:

In [ ]:
def area_of_square(side):
    if side < 0:
        raise ValueError("side must be non-negative")
    return side**2


def test_area_of_square_rejects_negative_sides():
    with pytest.raises(ValueError):
        area_of_square(-1)


test_area_of_square_rejects_negative_sides()   # pytest would call this for you; here it passes silently

### Organising/grouping tests

As you start working with `pytest` for testing, it's helpful to know that organizing your tests well becomes increasingly important as your code base grows. One approach is to define a class object in your test files (its name must start with `Test`, and it must not have an `__init__` method, for `pytest` to collect its tests), which may seem unnecessary for small pieces of code but pays off when managing larger projects. Structuring your tests in classes allows you to group related tests logically and keep your codebase clean and scalable.

A basic example of this is in `files/grouped_tests/test_groups.py`. From the `files/grouped_tests` folder, we can execute only the 'math operations' via
```python
pytest -k TestMathOperations
```
or
```python
pytest test_groups.py::TestMathOperations
```
and only the string operations via
```python
pytest -k TestStringOperations
```
and so forth.

### Useful `pytest` command-line options

A few commonly used flags you'll want to remember:

| Option | Meaning |
|---------|----------|
| `-v` | Verbose output — shows each test name and result |
| `-q` | Quiet mode — minimal output |
| `-x` | Stop after the first failure |
| `--maxfail=N` | Stop after *N* failures |
| `-s` | Disable output capture (so `print()` statements are shown) |
| `--pdb` | Drop into the debugger when a test fails or raises an exception |
| `--trace` | Drop into the debugger at the start of each test |
| `--lf` | Re-run only the tests that failed last time |
| `--doctest-modules` | Also run the examples in docstrings (see `doctest` below) |
| `--cov=<module>` | Measure code coverage (from `pytest-cov`; see below) |
| `--tb=short` | Compact tracebacks (useful for large test suites) |
| `--tb=long` | Full tracebacks |
| `--disable-warnings` | Hide warning messages |
| `-k "pattern"` | Run only tests whose names match a substring or pattern |
| `-m "marker"` | Run tests with a specific marker (e.g. `@pytest.mark.slow`) |

**Tip:** To see all available options, run:
```bash
pytest --help
```
or the shorthand
```
pytest -h
```

<div class="alert alert-info">

<h3>Exercise: <code>Test classes</code></h3>

<p>The folder <b>files/l4mod</b> contains the module <b>l4mod.py</b>: a Gaussian elimination routine, <code>gauss</code>, and two helpers, <code>matmul</code> and <code>zeromat</code>. Review the code, then (within the <b>l4mod</b> folder) create a test file containing 2 test classes:</p>

<ol>
    <li> <code>class TestUnits</code>, for <code>matmul</code> and <code>zeromat</code></li>
    <li> <code>class TestModule</code>, for <code>gauss</code></li>
</ol>

<p>Add some appropriate tests to each class. Compare floating-point results with a tolerance, and include a test that uses <code>pytest.raises</code>. Note that you can run the tests contained within a particular <code>class</code> via, e.g., <code>pytest test_&lt;name&gt;.py::TestUnits</code>. Why not also make use of the <a href="https://docs.pytest.org/en/stable/how-to/parametrize.html">pytest.mark.parametrize</a> decorator?</p>

<p><b>Read the docstrings carefully.</b> <code>gauss</code> promises to accept NumPy arrays as well as lists of lists: make sure your tests check every promise it makes, including with a matrix whose rows have to be swapped during the elimination. If a test fails, use <code>pytest --pdb</code> to find out why, fix <code>gauss</code>, and keep the test.</p>

<p>Note: Whilst separating tests in this way could be considered 'overkill' for this small piece of code, being able to apply these skills is important since larger projects organise their tests in this way! Take a look at the NumPy test suite <a href="https://github.com/numpy/numpy/tree/main/numpy/tests">here</a>.</p>

</div>

### The `doctest` module

The module [doctest](https://docs.python.org/3/library/doctest.html), from the standard Python library, provides a simple way to include code which is both a test and documentation of an example of the use of your code.

To write a test, one simply copies the input and output that one would see _in the vanilla python interpreter_ pretty much identically into a docstring, whether for a function or module. Consider `files/docstring_test.py`:

```python
def mean(x):
    """Mean of a list of numbers.

    >>> mean([1, 5, 9])
    5.0
    """
    return sum(x)/len(x)


if __name__ == "__main__":
    import doctest
    doctest.testmod()

```

In this case we can run the test by calling the module as a script:
```bash
python docstring_test.py
```

If the test succeeds it silently returns a successful exit code. If the test fails (e.g. if we change the expected output in the example from 5.0 to 3.0) then an error message is printed, looking like the following:

```bash
**********************************************************************
File "docstring_test.py", line 6, in __main__.mean
Failed example:
    mean([1, 5, 9])
Expected:
    3.0
Got:
    5.0
**********************************************************************
1 items had failures:
   1 of   1 in __main__.mean
***Test Failed*** 1 failures.
```

Doctests can also be run in Python on plain text files as

```python
import doctest
doctest.testfile("example.txt")
```
or from the command line as

```bash
python -m doctest example.txt
```

The same command works on Python modules (`python -m doctest -v docstring_test.py`), so the `if __name__ == "__main__":` block is optional. `pytest` can also run doctests alongside your other tests: `pytest --doctest-modules` collects the examples in every module's docstrings.

<div class="alert alert-info">

<h3>Exercise: <code>doctest</code></h3>

<p>For each of the functions in <code>l4mod</code>, complete its docstring ensuring a testable usage example is present. Use the doctest module, or `pytest --doctest-modules` together with your tests from the previous exercise, to ensure these 'usage examples' pass/fail as expected.</p>

</div>

### Code coverage

In general, it is best practice to ensure that your tests exercise every line of your code at least once. This is especially true for interpreted languages like Python, which only checks the syntax of a file (i.e its "grammar") when it is first loaded, but doesn't check that the meaning makes sense.

That means you can write code like the following in a module file and it will load without error

__example.py:__
```python
def f(x):
    return y
```

but trying to actually run it will error out, in this case with a NameError.

```python
import example
example.f(1)
```

returns
```
NameError: name 'y' is not defined
```

There is a Python package, `pytest-cov`, which adds coverage information to `pytest`. Once installed with `pip`, coverage information can be collected for a package or module called `mycoolproject` by calling `pytest` as 

```
python -m pytest --cov=mycoolproject .
```

This will generate output like

```
_______________ coverage: platform linux, python 3.14.7-final-0 ________________
Name                               Stmts   Miss  Cover
------------------------------------------------------
mycoolproject/__init__.py              4      0   100%
mycoolproject/analysis.py             33     24    27%
mycoolproject/live.py                 58     40    31%
mycoolproject/tests/test_example.py   11      1    91%
mycoolproject/validator.py            11      0   100%
------------------------------------------------------
TOTAL                                117     65    56%
```

Adding `--cov-report=term-missing` also lists the line numbers that were never run.

Note that while 100% code coverage is useful, it only means that the code has been run, not that every possible code logic branch has been tested. It is a useful, but neither necessary nor sufficient clue to healthy code.

**Exercise:** Why not try running this with your `l4mod.py` tests (and, with `--doctest-modules`, your doctests)? Which lines are not covered, and does it matter?

## Automatic documentation

Documentation is written for three broad groups of people:

1. Users ("I don't care how it does it, I care what it does.")
1. User-Developers ("I build what I need.")
1. Developers ("I make it, but I don't use it.")

These individuals have different needs from the documentation you provide to them. Users want to know what software can do, and how to run it for their own problems. Developers need to understand how code works, but not necessarily what it's intended for in a wider sense. User-developers need to understand both sides. Commercial software is generally written by pure developers, but a lot of scientific software (and many other small projects) is written by user-developers.

For all groups, the most important piece of documentation in Python is the docstring, since it remains with the code it applies to, and connects to the Python online help. For developers & user-developers, additional comments in the body of the source text may also be useful, however users will probably never choose to look there.

Although it is now generally acknowledged that the best place for living documentation is in the source code, near where it applies, since this gives the best probability of developers updating it when they change, it is still useful to maintain a proper (electronic) manual or write-up, both for ease of reference, and to give a general project overview.

Several tools have been developed to close this gap by automatically collating comments and function call signatures from the source code and converting it into a human readable document to label as a "manual". Perhaps the most famous open source, cross-language documentation tool is [Doxygen](http://www.doxygen.nl). However we'll look further at a tool called [Sphinx](https://www.sphinx-doc.org), which originated in the Python community and is the tool of choice on several large Python projects including SciPy, Django and [Python itself](https://docs.python.org/3/about.html).

### Sphinx

You can install the core Sphinx documentation generation tools with the command

```bash
pip install sphinx
```

Sphinx works by converting [reStructured text](https://docutils.sourceforge.io/docs/user/rst/quickstart.html), whether inside docstrings or in special files into HTML pages or PDF files.

The simplest use pattern is probably to use the automatic scanning tools to collect together all the docstrings into an index of APIs.

This requires creating two files, the first containing the configuration options for the Sphinx tools, and the second containing a skeleton reStructured text file into which to slot the generated documentation:

_`docs/conf.py`_:

```python
import sys
import os

## We're working in the ./docs directory, but need the package root in the path
## This command appends the directory one level up, in a cross-platform way. 
sys.path.insert(0, os.path.abspath(os.sep.join((os.curdir, '..'))))

project = 'MyCoolProject'
extensions = ['sphinx.ext.autodoc',
              'sphinx.ext.napoleon',
              'sphinx.ext.mathjax']
source_suffix = '.rst'
master_doc = 'index'
exclude_patterns = ['_build']
autoclass_content = "both"
```

_`docs/index.rst`_:

```rst
#############
mycoolproject
#############


A heading
---------

This is just example text, perhaps with mathematics like :math:`x^2`, **bold text** and *italics*.
It might also include citations [1]_ inline references to functions like :func:`my_func` or even whole code blocks::

    def my_example(a, b):
        """Do something!"""
        return a**b+1

.. automodule:: mycoolproject
  :members:
  
  
.. rubric:: References
[1] http://www.sphinx-doc.org/en/master/usage/restructuredtext/basics.html#citations
```


With this setup we can build a `html` version of the documentation with the command

```bash
sphinx-build -b html docs docs/html
```

If this is successful, you should be able to open `./docs/html/index.html` to see documentation automatically generated from the docstrings in your project. Sphinx also supports other output formats (for example LaTeX) with the `-b` flag. If you have LaTeX installed (see the note at the start of this lecture), a recipe to generate a `pdf` manual is

```bash
sphinx-build -b latex docs docs/tex
cd docs/tex
pdflatex MyCoolProject.tex
pdflatex MyCoolProject.tex
```

which will generate `MyCoolProject.pdf`. We run LaTeX twice to ensure that references and citations (including the index) are set correctly. 


<div class="alert alert-info">

<h3> Exercise: Autodocumenting your module</h3>

<ul><li> Create a <code>docs</code> directory inside the <code>l4mod</code> folder and add <code>conf.py</code> and <code>index.rst</code> files based on the ones given above (edit them as appropriate).</li>
<li> Run <code>sphinx-build</code> to generate <code>html</code> documentation for your project.</li>
<li> Try editing the <code>index.rst</code> file to add more text.</li>
<li> (Optional, needs LaTeX) Also build the <code>pdf</code> documentation.</li></ul>
    
</div>

**Note:** Worked solutions to this lecture's exercises, including a complete `l4mod` folder, will be released after the lecture.

## Further Reading

- The Pytest [documentation](https://docs.pytest.org/en/stable/).
- The Python documentation on [doctest](https://docs.python.org/3/library/doctest.html).
- A software carpentry [lecture](https://intermediate-and-advanced-software-carpentry.readthedocs.io/en/latest/testing-python.html) on testing.
- The official [Sphinx tutorial](https://www.sphinx-doc.org/en/master/tutorial/index.html)

## Self-check quiz

This lecture has a short multiple-choice [self-check quiz](quiz.ipynb) on its main points: open the notebook and click an option under each question. This lecture's exercises run alongside the material, so take the quiz once you have worked through it, and before you start any part of the coursework that relies on it. The answers, with explanations, will be released later so that you can check your own. The quiz is not assessed, but it is good preparation for the multiple-choice part of the consolidation test.
